# Analysis walkthrough

Exploration uses the training split only. Run `python run.py` from the repository root first. The experiment code lives in `src/`, so this notebook does not duplicate training logic.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from housing.data import download_data


In [ ]:
summary = json.loads((ROOT / 'reports/summary.json').read_text())
summary


## Data quality and leakage

Review missingness on training data; the test split is reserved for the frozen evaluation. Inspect the allowlists and preprocessing in `src/`.

In [ ]:
pd.read_csv(ROOT / 'reports/missing_train.csv').head(15)


In [ ]:
pd.read_csv(ROOT / 'reports/eda_train.csv', index_col=0).T.head(15)


## Model comparison

Model selection used the CV column, not the holdout metrics. Grid search scores are tuning scores and can be optimistic; they are not nested-CV estimates.

In [ ]:
pd.read_csv(ROOT / 'reports/model_comparison.csv')


In [ ]:
display(Image(filename=str(ROOT / 'images/model_comparison.png')))


## Error analysis

The final test predictions are diagnostic artifacts. Do not choose new hyperparameters based on these errors and then report the same test set as untouched.

In [ ]:
predictions = pd.read_csv(ROOT / 'reports/test_predictions.csv')
predictions.head(10)


In [ ]:
display(Image(filename=str(ROOT / 'images/target_distribution.png')))
display(Image(filename=str(ROOT / 'images/predictions_and_errors.png')))


In [ ]:
pd.read_csv(ROOT / 'reports/error_by_price_band.csv')


In [ ]:
pd.read_csv(ROOT / 'reports/worst_errors.csv')[['row_index', 'actual', 'predicted', 'absolute_error']]


In [ ]:
display(Image(filename=str(ROOT / 'images/feature_importance.png')))


## Interpretation

Read the conclusions and limitations in README. Permutation importance describes predictive dependence, not causality; correlated features can share importance. Any new experiment needs new validation planning.